In [3]:
import subprocess
result = subprocess.run(["find", "/usr/lib/jvm", "-name", "java", "-type", "f"], 
                        capture_output=True, text=True)
print(result.stdout)

/usr/lib/jvm/java-17-openjdk-amd64/bin/java



In [6]:
import os, random, datetime, subprocess

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
subprocess.run(["pip", "install", "pyspark", "-q"], check=True)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    DoubleType, IntegerType, TimestampType, StringType
)

# ---------------------------------------------------------------------------
# Config (replaces argparse for notebook use)
# ---------------------------------------------------------------------------
INPUT_PATH  = "/kaggle/working/synthetic_trips/"
OUTPUT_PATH = "/kaggle/working/agg_daily_revenue/"
MIN_DURATION = 1
MAX_DURATION = 180

# ---------------------------------------------------------------------------
# SparkSession — mirrors process_historical.py exactly
# ---------------------------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("nyc_taxi_historical_processing")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ---------------------------------------------------------------------------
# Generate synthetic data spanning 2009–2023 and write as Parquet
# (simulates reading real NYC TLC files via mergeSchema)
# ---------------------------------------------------------------------------
print("Generating synthetic 2009–2023 trip data...")
random.seed(42)
rows = []
base = datetime.datetime(2009, 1, 1, 6, 0, 0)
total_minutes = 15 * 365 * 24 * 60   # 2009 → 2023 = ~15 years

for _ in range(150_000):
    pickup   = base + datetime.timedelta(minutes=random.randint(0, total_minutes))
    duration = float(random.choice([-5, 0, random.randint(1, 180)]))
    dropoff  = pickup + datetime.timedelta(minutes=duration)
    dist     = float(round(random.choice([0.0, random.uniform(0.1, 25.0)]), 2))
    fare     = float(round(random.uniform(3, 80) if dist > 0 else 0.0, 2))
    tip      = float(round(fare * random.uniform(0, 0.3), 2))
    rows.append((
        pickup, dropoff,
        random.choice([0, 1, 2, 3, 4]),
        dist, fare, tip,
        float(round(fare + tip + 2.5, 2)),
        random.randint(1, 265),
        random.randint(1, 265),
        random.randint(1, 5),
    ))

raw_schema = StructType([
    StructField("tpep_pickup_datetime",  TimestampType()),
    StructField("tpep_dropoff_datetime", TimestampType()),
    StructField("passenger_count",       IntegerType()),
    StructField("trip_distance",         DoubleType()),
    StructField("fare_amount",           DoubleType()),
    StructField("tip_amount",            DoubleType()),
    StructField("total_amount",          DoubleType()),
    StructField("PULocationID",          IntegerType()),
    StructField("DOLocationID",          IntegerType()),
    StructField("payment_type",          IntegerType()),
])

raw_df = spark.createDataFrame(rows, raw_schema)
raw_df.write.mode("overwrite").parquet(INPUT_PATH)
print(f"Raw rows written: {raw_df.count():,}")

# ---------------------------------------------------------------------------
# 1. Read all Parquet files (mergeSchema handles column drift across years)
# ---------------------------------------------------------------------------
raw = spark.read.option("mergeSchema", "true").parquet(INPUT_PATH)

# ---------------------------------------------------------------------------
# 2. Staging + cleaning — mirrors stg_yellow_trips.sql exactly
# ---------------------------------------------------------------------------
staged = (
    raw
    .withColumn("pickup_datetime",
                F.col("tpep_pickup_datetime").cast(TimestampType()))
    .withColumn("dropoff_datetime",
                F.col("tpep_dropoff_datetime").cast(TimestampType()))
    .withColumn("passenger_count",
                F.col("passenger_count").cast(IntegerType()))
    .withColumn("pickup_location_id",
                F.col("PULocationID").cast(IntegerType()))
    .withColumn("dropoff_location_id",
                F.col("DOLocationID").cast(IntegerType()))
    .withColumn("payment_type",
                F.col("payment_type").cast(IntegerType()))
    .withColumn("trip_distance_miles",
                F.col("trip_distance").cast(DoubleType()))
    .withColumn("fare_amount",
                F.col("fare_amount").cast(DoubleType()))
    .withColumn("tip_amount",
                F.col("tip_amount").cast(DoubleType()))
    .withColumn("total_amount",
                F.col("total_amount").cast(DoubleType()))
    .withColumn("trip_duration_minutes",
                (F.unix_timestamp("dropoff_datetime") -
                 F.unix_timestamp("pickup_datetime")) / 60.0)
    # Same filter rules as int_trips_enriched.sql
    .filter(F.col("trip_distance_miles") > 0)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("passenger_count") > 0)
    .filter(F.col("trip_duration_minutes") >= MIN_DURATION)
    .filter(F.col("trip_duration_minutes") <= MAX_DURATION)
    .withColumn("year",        F.year("pickup_datetime"))
    .withColumn("month",       F.month("pickup_datetime"))
    .withColumn("pickup_date", F.to_date("pickup_datetime"))
    .select(
        "pickup_datetime", "dropoff_datetime", "trip_duration_minutes",
        "passenger_count", "trip_distance_miles",
        "pickup_location_id", "dropoff_location_id",
        "fare_amount", "tip_amount", "total_amount", "payment_type",
        "pickup_date", "year", "month"
    )
)

# ---------------------------------------------------------------------------
# 3. Broadcast join with zone lookup (265 zones — tiny table, no shuffle)
# ---------------------------------------------------------------------------
boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island", "EWR"]
zone_rows = [(i, f"Zone {i}", boroughs[i % 6]) for i in range(1, 266)]
zone_schema = StructType([
    StructField("location_id", IntegerType()),
    StructField("zone_name",   StringType()),
    StructField("borough",     StringType()),
])
zones_b = F.broadcast(
    spark.createDataFrame(zone_rows, zone_schema)
)
staged = staged.join(
    zones_b,
    staged.pickup_location_id == zones_b.location_id,
    "left"
).drop("location_id")

# ---------------------------------------------------------------------------
# 4. Cache and materialise
# ---------------------------------------------------------------------------
staged.cache()
clean_count = staged.count()
print(f"\nAfter filtering: {clean_count:,} rows")

# Row counts per year — confirms 2009-2023 coverage
print("\n── Trips per year ──")
staged.groupBy("year").count().orderBy("year").show(20, truncate=False)

# ---------------------------------------------------------------------------
# 5. Compute agg_daily_revenue — mirrors the dbt mart exactly
# ---------------------------------------------------------------------------
daily_revenue = (
    staged.groupBy("pickup_date", "year", "month")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips"),
        F.round(
            100.0 * F.sum("tip_amount") /
            F.nullif(F.sum("fare_amount"), F.lit(0)),
            2
        ).alias("tip_rate_pct"),
    )
)

print("\n── agg_daily_revenue sample (one day per year) ──")
daily_revenue.orderBy("year", "month", "pickup_date") \
    .dropDuplicates(["year"]).show(15, truncate=False)

# ---------------------------------------------------------------------------
# 6. Write partitioned Parquet output
# ---------------------------------------------------------------------------
(
    daily_revenue
    .repartition("year", "month")
    .write
    .mode("overwrite")
    .partitionBy("year", "month")
    .parquet(OUTPUT_PATH)
)

file_count = sum(len(f[2]) for f in os.walk(OUTPUT_PATH))
print(f"\nDone. Output written to {OUTPUT_PATH}")
print(f"Parquet files written: {file_count}")

staged.unpersist()
spark.stop()

Generating synthetic 2009–2023 trip data...


26/07/12 16:40:28 WARN TaskSetManager: Stage 0 contains a task of very large size (2364 KiB). The maximum recommended task size is 1000 KiB.
26/07/12 16:40:30 WARN TaskSetManager: Stage 1 contains a task of very large size (2364 KiB). The maximum recommended task size is 1000 KiB.


Raw rows written: 150,000

After filtering: 20,148 rows

── Trips per year ──
+----+-----+
|year|count|
+----+-----+
|2009|1367 |
|2010|1381 |
|2011|1276 |
|2012|1347 |
|2013|1343 |
|2014|1431 |
|2015|1349 |
|2016|1280 |
|2017|1340 |
|2018|1361 |
|2019|1354 |
|2020|1361 |
|2021|1332 |
|2022|1319 |
|2023|1307 |
+----+-----+


── agg_daily_revenue sample (one day per year) ──
+-----------+----+-----+-----------+----------+--------+----------+------------+
|pickup_date|year|month|total_trips|total_fare|avg_fare|total_tips|tip_rate_pct|
+-----------+----+-----+-----------+----------+--------+----------+------------+
|2018-01-01 |2018|1    |3          |213.91    |71.3    |53.44     |24.98       |
|2015-01-01 |2015|1    |2          |101.43    |50.72   |12.53     |12.35       |
|2023-01-01 |2023|1    |7          |269.84    |38.55   |45.5      |16.86       |
|2022-01-01 |2022|1    |5          |283.09    |56.62   |42.03     |14.85       |
|2013-01-01 |2013|1    |3          |56.56     |18.85   |


Done. Output written to /kaggle/working/agg_daily_revenue/
Parquet files written: 362
